In [1]:
# Library
import requests
import pandas as pd
import numpy as np
import time
import json
import os
import getpass
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Folder
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/clean", exist_ok=True)
os.makedirs("../report", exist_ok=True)

# Settingan Visual untuk Plot 
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

print("Semua library berhasil diimpor.")

Semua library berhasil diimpor.


In [2]:
# 1. Konfigurasi Github Token dan Target Repo
# Konfigurasi Input Token 
print("Masukkan GitHub Personal Access Token Anda:")
GITHUB_TOKEN = getpass.getpass()

# Konfigurasi target repo
REPO = "pandas-dev/pandas"
HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"}

print("\n Konfigurasi selesai.")
print(f"  Target repo : {REPO}")
print(f"  Token       : {'*' * 20}")

Masukkan GitHub Personal Access Token Anda:

 Konfigurasi selesai.
  Target repo : pandas-dev/pandas
  Token       : ********************


In [3]:
# 2. Data Fetching dari Github API untuk PR dan Issues
def fetch_github_data(endpoint, params, max_pages=20):
    all_data = []
    url = f"https://api.github.com/repos/{REPO}/{endpoint}"
    print(f"Mengambil data dari endpoint: /{endpoint}")
    
    for page in range(1, max_pages + 1):
        params["page"] = page
        params["per_page"] = 100
        
        response = requests.get(url, headers=HEADERS, params=params)
        
        if response.status_code != 200:
            print(f"Berhenti di halaman {page}. Status code: {response.status_code}")
            break
            
        data = response.json()
        if not data:
            print(f"Semua data sudah terambil.Selesai di halaman {page - 1}.")
            break 
            
        all_data.extend(data)
        print(f"Halaman {page}: {len(data)} item  |  Total terkumpul: {len(all_data)}")
        time.sleep(1) 

    print(f"\nSelesai! Total data dari /{endpoint}: {len(all_data)} item\n")
    return all_data

# A. Fetch Pull Requests (RQ1)
pr_params = {"state": "closed", "sort": "created", "direction": "desc","since": "2021-01-01T00:00:00Z"}
raw_prs = fetch_github_data("pulls", pr_params, max_pages=200) 
with open("../data/raw/raw_prs.json", "w") as f: json.dump(raw_prs, f) # Simpan data mentah sebagai backup
print(f"Data berhasil disimpan ke: {"../data/raw/raw_prs.json"}")
print(f"\nTotal raw PR tersedia: {len(raw_prs)}")

# B. Fetch Issues (RQ2 & RQ3)
print("Memulai fetch data issues untuk data pre dan post Pandas 2.0.\n")
issue_params = {"state": "closed", "sort": "created", "direction": "desc", "since": "2021-01-01T00:00:00Z"}
raw_issues = fetch_github_data("issues", issue_params, max_pages=200)

with open("../data/raw/raw_issues.json", "w") as f: json.dump(raw_issues, f) # Simpan data mentah sebagai backup
print(f"Data berhasil disimpan ke: {"../data/raw/raw_issues.json"}")
print(f"\nTotal raw Issues tersedia: {len(raw_issues)}")

print(f"\n Selesai! Total PR ditarik: {len(raw_prs)} | Total Issue ditarik: {len(raw_issues)}")

Mengambil data dari endpoint: /pulls
Halaman 1: 100 item  |  Total terkumpul: 100
Halaman 2: 100 item  |  Total terkumpul: 200
Halaman 3: 100 item  |  Total terkumpul: 300
Halaman 4: 100 item  |  Total terkumpul: 400
Halaman 5: 100 item  |  Total terkumpul: 500
Halaman 6: 100 item  |  Total terkumpul: 600
Halaman 7: 100 item  |  Total terkumpul: 700
Halaman 8: 100 item  |  Total terkumpul: 800
Halaman 9: 100 item  |  Total terkumpul: 900
Halaman 10: 100 item  |  Total terkumpul: 1000
Halaman 11: 100 item  |  Total terkumpul: 1100
Halaman 12: 100 item  |  Total terkumpul: 1200
Halaman 13: 100 item  |  Total terkumpul: 1300
Halaman 14: 100 item  |  Total terkumpul: 1400
Halaman 15: 100 item  |  Total terkumpul: 1500
Halaman 16: 100 item  |  Total terkumpul: 1600
Halaman 17: 100 item  |  Total terkumpul: 1700
Halaman 18: 100 item  |  Total terkumpul: 1800
Halaman 19: 100 item  |  Total terkumpul: 1900
Halaman 20: 100 item  |  Total terkumpul: 2000
Halaman 21: 100 item  |  Total terkumpul:

In [6]:
# 3. Data Cleaning untuk PR
PANDAS_V2_DATE = pd.Timestamp("2023-04-03") 

pr_list = []
jumlah_draft_dilewati = 0
jumlah_tanpa_closed = 0
jumlah_anomali = 0
jumlah_diluar_range = 0

for pr in raw_prs:
    if pr.get("draft") == True: 
        jumlah_draft_dilewati += 1
        continue 
    
    created   = pd.Timestamp(pr["created_at"])
   
    start_bound = pd.Timestamp("2021-01-01", tz="UTC")
    end_bound = pd.Timestamp("2025-12-31 23:59:59", tz="UTC")

    if not (start_bound <= created <= end_bound):
        jumlah_diluar_range += 1
        continue
    
    closed_at = pr.get("closed_at")
    merged_at = pr.get("merged_at")
    
    if closed_at is None:
        jumlah_tanpa_closed += 1
        continue

    closed = pd.Timestamp(closed_at)
    days   = (closed - created).total_seconds() / 86400

    if days <= 0:
        jumlah_anomali += 1
        continue

    is_merged = True if merged_at else False

    pr_list.append({
        "number"         : pr["number"],
        "created_at"     : created,
        "merged"         : is_merged,
        "outcome"        : 1 if is_merged else 0,  
        "resolution_days": round(days, 2)
    })

df_prs = pd.DataFrame(pr_list)

print("HASIL CLEANING DATA PULL REQUEST")
print(f"Raw PR masuk              : {len(raw_prs)}")
print(f"Draft PR dilewati         : {jumlah_draft_dilewati}")
print(f"PR tanpa closed_at        : {jumlah_tanpa_closed}")
print(f"Data anomali (days <= 0)  : {jumlah_anomali}")
print(f"Data di luar range tanggal: {jumlah_diluar_range}")
print(f"───────────────────────────────")
print(f"PR valid (df_prs)         : {len(df_prs)}")
print(f"PR Merged   (k)           : {df_prs['merged'].sum()}")
print(f"PR Rejected (m)           : {(~df_prs['merged']).sum()}")
print()
df_prs.head()

HASIL CLEANING DATA PULL REQUEST
Raw PR masuk              : 20000
Draft PR dilewati         : 566
PR tanpa closed_at        : 0
Data anomali (days <= 0)  : 0
Data di luar range tanggal: 3725
───────────────────────────────
PR valid (df_prs)         : 15709
PR Merged   (k)           : 13038
PR Rejected (m)           : 2671



,number,created_at,merged,outcome,resolution_days
0,63525,2025-12-31 21:19:12+00:00,True,1,0.93
1,63524,2025-12-31 19:57:20+00:00,False,0,29.94
2,63523,2025-12-31 17:26:51+00:00,True,1,0.08
3,63522,2025-12-31 16:39:28+00:00,True,1,1.12
4,63520,2025-12-31 06:37:27+00:00,True,1,1.53


In [7]:
# 4. Data Cleaning untuk Issues
issue_list = []
jumlah_pr_dilewati  = 0
jumlah_anomali_iss  = 0
jumlah_diluar_range = 0

for issue in raw_issues:
    if "pull_request" in issue:
        jumlah_pr_dilewati += 1
        continue
    
    created = pd.Timestamp(issue["created_at"])

    start_bound = pd.Timestamp("2021-01-01", tz="UTC")
    end_bound = pd.Timestamp("2025-12-31 23:59:59", tz="UTC")

    if not (start_bound <= created <= end_bound):
        jumlah_diluar_range += 1
        continue

    labels = [label["name"].lower() for label in issue.get("labels", [])]
    is_bug = any("bug" in label for label in labels)
    
    closed_at = issue.get("closed_at")
    if closed_at is None: continue

    closed = pd.Timestamp(closed_at)
    days   = (closed - created).total_seconds() / 86400

    if days <= 0:
        jumlah_anomali_iss += 1
        continue

    issue_list.append({
        "number"         : issue["number"],
        "title"          : issue.get("title", ""),  
        "type"           : "bug" if is_bug else "other",
        "created_at"     : created,
        "resolution_days": round(days, 2)
    })

df_issues = pd.DataFrame(issue_list)

# Buat dataframe khusus bug per bulan untuk Poisson (RQ2)
df_issues["yearmonth"] = df_issues["created_at"].dt.to_period("M")
df_bugs = df_issues[df_issues["type"] == "bug"].copy()
monthly_bugs = df_bugs.groupby("yearmonth").size().reset_index(name="bug_count")
monthly_bugs["timestamp"] = monthly_bugs["yearmonth"].dt.to_timestamp()
monthly_bugs["period"]    = monthly_bugs["timestamp"].apply(
    lambda x: "post_v2" if x >= PANDAS_V2_DATE else "pre_v2"
)

print("HASIL CLEANING DATA ISSUES")
print(f"Raw issues masuk               : {len(raw_issues)}")
print(f"PR yang dilewati               : {jumlah_pr_dilewati}")
print(f"Anomali (days <= 0)            : {jumlah_anomali_iss}")
print(f"Di luar rentang tanggal        : {jumlah_diluar_range}")
print(f"  ───────────────────────────────")
print(f"Issue valid (df_issues)        : {len(df_issues)}")
print(f"Tipe Bug                       : {len(df_bugs)}")
print(f"Tipe Lain                      : {len(df_issues) - len(df_bugs)}")
print(f"Bulan data bug (monthly_bugs)  : {len(monthly_bugs)} bulan")
print()
df_issues.head()

C:\Users\Darren CW\AppData\Local\Temp\ipykernel_27680\559215100.py:45: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_issues["yearmonth"] = df_issues["created_at"].dt.to_period("M")


HASIL CLEANING DATA ISSUES
Raw issues masuk               : 9900
PR yang dilewati               : 7456
Anomali (days <= 0)            : 0
Di luar rentang tanggal        : 183
  ───────────────────────────────
Issue valid (df_issues)        : 2261
Tipe Bug                       : 1129
Tipe Lain                      : 1132
Bulan data bug (monthly_bugs)  : 29 bulan



,number,title,type,created_at,resolution_days,yearmonth
0,63521,BUG: comparison of categoricals with NA,bug,2025-12-31 11:01:37+00:00,0.13,2025-12
1,63518,BUG: pd.Grouper converts PyArrow timestamp to ...,bug,2025-12-31 02:09:34+00:00,1.72,2025-12
2,63489,ENH: drop function should take kwarg rows,other,2025-12-26 23:06:58+00:00,0.59,2025-12
3,63484,BUG: pd.api.types.pandas_dtype raise different...,bug,2025-12-25 12:38:52+00:00,7.81,2025-12
4,63477,QST: Error while building document,other,2025-12-24 03:33:23+00:00,136.52,2025-12


In [8]:
# 5. Simpan Dataset Bersih ke CSV
df_prs.to_csv("../data/clean/prs_clean.csv", index=False)
df_issues.to_csv("../data/clean/issues_full.csv", index=False)
monthly_bugs.to_csv("../data/clean/monthly_bugs.csv", index=False)

print("Data berhasil dibersihkan dan disimpan ke folder clean")
print(f"prs_clean.csv:{len(df_prs)} baris")
print(f"issues_full.csv:{len(df_issues)} baris")
print(f"monthly_bugs.csv:{len(monthly_bugs)} baris")

Data berhasil dibersihkan dan disimpan ke folder clean
prs_clean.csv:15709 baris
issues_full.csv:2261 baris
monthly_bugs.csv:29 baris


In [ ]:
# 6. Statistik Deskriptif
print("STATISTIK DESKRIPTIF — PULL REQUEST (df_prs)")
print(df_prs[["outcome", "resolution_days"]].describe().round(2))

print()
print("STATISTIK DESKRIPTIF — ISSUES (df_issues)")
print(df_issues[["resolution_days"]].describe().round(2))

print()
print("STATISTIK DESKRIPTIF — BUG PER BULAN (monthly_bugs)")
print(monthly_bugs[["bug_count"]].describe().round(2))

print()
print("DISTRIBUSI TIPE ISSUE")
print(df_issues["type"].value_counts().to_string())

In [ ]:
# 7. EDA RQ1: Proporsi PR MERGED vs REJECTED

# Hitung variabel kunci untuk RQ1
k = int(df_prs["merged"].sum())       
m = int((~df_prs["merged"]).sum())    
n = len(df_prs)                       
theta_preview = round(k / n, 4)       

print("RQ1: OUTCOME PULL REQUEST")
print(f"Total PR (n)       : {n}")
print(f"PR Merged (k)      : {k}")
print(f"PR Rejected (m)    : {m}")
print(f"Preview θ̂ = k/n    : {theta_preview}")
print()

# Visualitation RQ1
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(
    data=df_prs,
    x="merged",
    palette={"False":"#EF5350", "True":"#4CAF50"},
    ax=axes[0]
)
axes[0].set_title("Jumlah PR: Merged vs Rejected", fontsize=12)
axes[0].set_xlabel("Outcome PR")
axes[0].set_ylabel("Jumlah PR")
axes[0].set_xticklabels(["Rejected (0)", "Merged (1)"])

for bar in axes[0].patches:
    height = int(bar.get_height())
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        height + max(n * 0.005, 3),
        f"{height:,}",
        ha="center", va="bottom",
        fontsize=11, fontweight="bold"
    )

pie_labels = [
    f"Merged\n{k:,} ({theta_preview * 100:.1f}%)",
    f"Rejected\n{m:,} ({(1 - theta_preview) * 100:.1f}%)"
]
axes[1].pie(
    [k, m],
    labels=pie_labels,
    colors=["#4CAF50", "#EF5350"],
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
axes[1].set_title("Proporsi PR Merged vs Rejected", fontsize=12)

plt.suptitle(
    f"EDA — RQ1: Outcome Pull Request (Total: {n:,} PR)",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
# 8. EDA RQ2: Report Jumlah Bug Pre vs Post Pandas v2.0
pre_data = monthly_bugs[monthly_bugs["period"] == "pre_v2"]["bug_count"]
post_data = monthly_bugs[monthly_bugs["period"] == "post_v2"]["bug_count"]

lam_pre = round(pre_data.mean(), 4)   
lam_post = round(post_data.mean(), 4)  

print("RQ2: TREN BUG REPORT PRE vs POST PANDAS 2.0")
print(f"Bulan data Pre  Pandas 2.0  : {len(pre_data)} bulan")
print(f"Bulan data Post Pandas 2.0  : {len(post_data)} bulan")
print(f"Preview λ̂₁ (pre  v2.0)      : {lam_pre} bug/bulan")
print(f"Preview λ̂₂ (post v2.0)      : {lam_post} bug/bulan")
print(f"Selisih rata-rata           : {abs(lam_post - lam_pre):.4f} bug/bulan")
print()

# Visualitation RQ2
fig, axes = plt.subplots(2, 1, figsize=(13, 10))

sns.lineplot(
    data=monthly_bugs,
    x="timestamp", y="bug_count",
    hue="period",
    palette={"pre_v2": "#1E88E5", "post_v2": "#FF7043"},
    marker="o", markersize=5,
    ax=axes[0]
)
axes[0].axvline(
    x=PANDAS_V2_DATE, color="red", linestyle="--", linewidth=2,
    label=f"Pandas 2.0 Rilis ({PANDAS_V2_DATE.strftime('%d %b %Y')})"
)
axes[0].axhline(
    y=lam_pre, color="#1E88E5", linestyle=":", linewidth=1.5,
    label=f"Rata-rata Pre v2.0: {lam_pre:.1f} bug/bulan"
)
axes[0].axhline(
    y=lam_post, color="#FF7043", linestyle=":", linewidth=1.5,
    label=f"Rata-rata Post v2.0: {lam_post:.1f} bug/bulan"
)
axes[0].set_title("Tren Laporan Bug per Bulan (Pre vs Post Pandas 2.0)", fontsize=12)
axes[0].set_xlabel("Bulan")
axes[0].set_ylabel("Jumlah Bug")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(fontsize=9)

plot_box = pd.DataFrame({
    "Bug Count": list(pre_data) + list(post_data),
    "Periode"  : ["Pre v2.0"] * len(pre_data) + ["Post v2.0"] * len(post_data)
})
sns.boxplot(
    data=plot_box,
    x="Periode", y="Bug Count",
    palette={"Pre v2.0": "#1E88E5", "Post v2.0": "#FF7043"},
    ax=axes[1]
)
axes[1].set_title("Perbandingan Distribusi Bug per Bulan: Pre vs Post Pandas 2.0", fontsize=12)
axes[1].set_xlabel("Periode")
axes[1].set_ylabel("Jumlah Bug per Bulan")

plt.suptitle(
    "EDA — RQ2: Analisis Tingkat Bug Report Sebelum dan Sesudah Pandas 2.0",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()
